In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import proplot as pplt
sys.path.insert(0,'..')
from scripts.utils import Config,load_stats
from scripts.models.sr.train import load_features,subsample_timesteps
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
CONFIG    = Config()
STATS     = load_stats(CONFIG.splitsdir)
RUNNAME   = 'sr_bl'
RUNCONFIG = CONFIG.sr['runs'][RUNNAME]
SUBFRAC   = CONFIG.sr['subsetfrac']
SEED      = CONFIG.sr['seeds'][0]
BINS      = np.array([0,1e-4,1e-2,1e-1,1,10,100,np.inf])
BINLABELS = ['Dry (≤1e-4)','(1e-4, 1e-2]','(1e-2, 0.1]','(0.1, 1]','(1, 10]','(10, 100]','>100']

In [ ]:
def load_fit_data():
    xtrain,ytrain,reftrain,trainmask = load_features('train',RUNCONFIG,CONFIG,timeoffset=0)
    xvalid,yvalid,_,validmask        = load_features('valid',RUNCONFIG,CONFIG,timeoffset=int(reftrain.sizes['time']))
    xfit = pd.concat([xtrain[trainmask],xvalid[validmask]]).reset_index(drop=True)
    yfit = np.concatenate([ytrain[trainmask],yvalid[validmask]])
    return xfit,yfit

def to_mm(y):
    return np.expm1(np.asarray(y)*STATS[f'{CONFIG.targetvar}_std']+STATS[f'{CONFIG.targetvar}_mean'])

def summary_stats(tp):
    return {
        'Samples':f'{len(tp)}',
        'Dry (≤1e-4 mm)':f'{np.mean(tp<=1e-4):.2%}',
        'Mean':f'{np.mean(tp):.3f}',
        'Median':f'{np.median(tp):.3f}',
        '90th Percentile':f'{np.percentile(tp,90):.3f}',
        '99th Percentile':f'{np.percentile(tp,99):.3f}',
        'Max':f'{np.max(tp):.3f}'}

def bin_fracs(tp,bins):
    counts = np.bincount(np.digitize(tp,bins[1:-1],right=True),minlength=len(bins)-1)
    return counts/counts.sum()

In [ ]:
xfit,yfit = load_fit_data()
xsub,ysub = subsample_timesteps(xfit,yfit,SUBFRAC,SEED,STATS)
full   = to_mm(yfit)
subset = to_mm(ysub)

In [ ]:
fullstats   = summary_stats(full)
subsetstats = summary_stats(subset)
subsetstats['Samples'] = f'{len(subset)} ({len(subset)/len(full):.3%})'
pd.DataFrame({'Full':fullstats,'Subset':subsetstats})

In [ ]:
pd.DataFrame({'Full Bin Fraction':bin_fracs(full,BINS),'Subset Bin Fraction':bin_fracs(subset,BINS)},index=BINLABELS)

In [ ]:
wetfull   = full[full>1e-4]
wetsubset = subset[subset>1e-4]
histbins  = np.logspace(-4,2,80)

fig,ax = pplt.subplots(refwidth=4,refheight=2)
ax.format(xlabel='Total Precipitation (mm)',xscale='log',xformatter='log',
          ylabel='Density',title='Wet-Point Precipitation Distribution',grid=True)
ax.hist(wetfull,bins=histbins,density=True,histtype='step',linewidth=1.5,color='#D42028',label='Full')
ax.hist(wetsubset,bins=histbins,density=True,histtype='step',linewidth=1.5,color='#5BA7DA',label='Subset')
ax.legend(loc='ur',ncols=1)
pplt.show()

In [ ]:
xidx = np.arange(len(BINLABELS))

fig,ax = pplt.subplots(refwidth=4,refheight=2)
ax.format(title='Rain Bin Fractions',xlabel='Precipitation Bin (mm)',xlocator=xidx,xformatter=BINLABELS,xrotation=45,
          ylabel='Fraction of Samples')
ax.plot(xidx,bin_fracs(full,BINS),color='#D42028',marker='o',label='Full')
ax.plot(xidx,bin_fracs(subset,BINS),color='#5BA7DA',marker='o',label='Subset')
ax.legend(loc='ur',ncols=1)
pplt.show()